# Power Outage Forecasting
This notebook is a demo for how to load, process, train on, and produce prediction files with the provided dataset.
You should not be confined by this notebook, but highly encouraged to produce your own pipeline or models based on what you learned from the course.

The format of submission files, however, is VERY strict. Student have to follow the provided tempaltes and the submitted files must pass the sanity tests in the last section.

Note the test files loaded in this notebook contain random noise. They only serve the purpose to dmostrate how evalution will be done. The actual test file has been hidden.

It is encouraged to run the starter notebook in Google Colab for easy of debug and implementation. You could also use your local Python notebook.
There is a chance that you run into out of memory when running this demo notebook in Colab free tier. In that case, you could either apply for free Google Colab Pro subscription as student and turn on high RAM, or only run one of the three models (all of the SARIMAX models, Seq2seq model for 24h, or Seq2seq model for 48h) included in the notebook.

## Prerequisites & Installation

Before running this demo notebook, ensure you have the required packages installed:

- `xarray` - for reading NetCDF files
- `netCDF4` - backend for xarray
- `numpy` - numerical operations
- `pandas` - data manipulation
- `matplotlib` - visualization
- `torch` - deep learning (PyTorch)
- `statsmodels` - SARIMAX models
- `tqdm` - progress bars


In [12]:
# Install dependencies
import subprocess, sys, platform

def is_conda():
    return "conda" in sys.prefix or "anaconda" in sys.prefix

if is_conda():
    print("Conda environment detected.")
    print("Installing netCDF4 and PyTorch via conda (this may take a minute)...")
    conda_pkgs = ["netcdf4", "pytorch"]
    # cpuonly meta-package only exists on Linux/Windows, not macOS
    if platform.system() != "Darwin":
        conda_pkgs.append("cpuonly")
    subprocess.check_call(
        ["conda", "install", "-y", "-c", "conda-forge", "-c", "pytorch"] + conda_pkgs
    )
    print("Done.")

# Install remaining packages via pip
!pip install -r requirements.txt

# For non-conda environments, install PyTorch via pip with official index
if not is_conda():
    !pip install torch --index-url https://download.pytorch.org/whl/cpu

Conda environment detected.
Installing netCDF4 and PyTorch via conda (this may take a minute)...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - pytorch
 - defaults
Platform: osx-arm64
Solving environment: done

# All requested packages already installed.

Done.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
# Check dependencies from requirements.txt
import sys

# Map package names to their import names (only for cases where they differ)
IMPORT_NAME_MAP = {
    "python-dotenv": "dotenv",
    "netCDF4": "netCDF4",
}

missing_packages = []
with open("requirements.txt") as f:
    for line in f:
        package = line.strip()
        if not package or package.startswith("#"):
            continue
        import_name = IMPORT_NAME_MAP.get(package, package)
        try:
            __import__(import_name)
            print(f"✓ {package} is installed")
        except ImportError:
            print(f"✗ {package} is NOT installed")
            missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
    print("Run: pip install -r requirements.txt")
else:
    print("\n✓ All required packages are installed!")

✓ numpy is installed
✓ pandas is installed
✓ xarray is installed
✓ netCDF4 is installed
✓ matplotlib is installed
✓ statsmodels is installed
✓ tqdm is installed
✓ wandb is installed
✓ python-dotenv is installed

✓ All required packages are installed!


## 1. Configuration & Setup

Define all meta variables and parameters at the start for easy configuration.

In [14]:
# ============== DATA SETUP ==============
# train.nc is too large for GitHub. Please download it manually:
#   1. Download train.nc from the shared Google Drive / Canvas
#   2. Place it in the data/ folder: data/train.nc
# The other data files (test_24h_demo.nc, test_48h_demo.nc) are already in data/
# ==========================================

# ============== META VARIABLES ==============
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Data paths
DATA_DIR = "data/"
RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.nc")

# The test_24h_demo.nc and test_48h_demo.nc are just demo files, so the outages in those files are just noise
# TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h.nc")
# TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h.nc")
TEST_24H_PATH = os.path.join(DATA_DIR, "test_24h_demo.nc")
TEST_48H_PATH = os.path.join(DATA_DIR, "test_48h_demo.nc")

# Model parameters
VALIDATION_SPLIT = 0.2  # Use last 20% of training data for validation

# SARIMAX parameters
SARIMAX_ORDER = (1, 0, 1)  # (p, d, q)

# Seq2Seq parameters
SEQ_LEN = 24       # Lookback window (hours) for the seq2seq model
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
HIDDEN_DIM = 64
NUM_LAYERS = 1

# Set device for PyTorch
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============== WANDB SETUP ==============
import wandb
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()  # Load .env file

WANDB_USERNAME = os.getenv("WANDB_USERNAME")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")  # Team name on wandb

if not WANDB_USERNAME or not WANDB_API_KEY or not WANDB_ENTITY:
    print("⚠️  WANDB credentials not fully configured in .env file!")
    print("   Required: WANDB_USERNAME, WANDB_API_KEY, WANDB_ENTITY")
    print("   Copy .env.example to .env and fill in your credentials.")
    print("   Notebook will still run, but experiments won't be logged to W&B.\n")
    WANDB_ENABLED = False
else:
    wandb.login(key=WANDB_API_KEY)
    run_name = f"{WANDB_USERNAME}_{datetime.now().strftime('%m%d_%H%M%S')}"
    wandb.init(
        project="MLPS-Power-Outage",
        entity=WANDB_ENTITY,
        name=run_name,
        config={
            "random_seed": RANDOM_SEED,
            "validation_split": VALIDATION_SPLIT,
            "sarimax_order": SARIMAX_ORDER,
            "seq_len": SEQ_LEN,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "device": DEVICE,
        },
    )
    WANDB_ENABLED = True
    print(f"✓ W&B initialized: run={run_name}, entity={WANDB_ENTITY}")

print(f"\nConfiguration loaded successfully!")
print(f"Random Seed: {RANDOM_SEED}")
print(f"Device: {DEVICE}")
print(f"Data Directory: {DATA_DIR}")
print(f"Results Directory: {RESULTS_DIR}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/gongwangrui/.netrc


CommError: Error uploading run: returned error 404: {"data":{"upsertBucket":null},"errors":[{"message":"entity your_team_name not found during upsertBucket","path":["upsertBucket"]}]}

## 2. Data Loading

Load the NetCDF datasets and explore their structure.

In [ ]:
# Load datasets
ds_train = xr.open_dataset(TRAIN_PATH)
ds_test_24h = xr.open_dataset(TEST_24H_PATH)
ds_test_48h = xr.open_dataset(TEST_48H_PATH)

In [ ]:
# Extract basic information
train_timestamps = pd.to_datetime(ds_train.timestamp.values)
locations = list(ds_train.location.values)
weather_features = list(ds_train.feature.values) if 'feature' in ds_train.dims else []

print(f"Training Period: {train_timestamps.min()} to {train_timestamps.max()}")
print(f"Number of Timestamps: {len(train_timestamps)}")
print(f"Number of Locations: {len(locations)}")
print(f"Locations: {locations}")
print(f"\nWeather Features ({len(weather_features)}): {weather_features}")

# Extract outage data
outage_data = ds_train.out.transpose("timestamp", "location").values.astype(float)
print(f"\nOutage Data Shape: {outage_data.shape} (timestamps x locations)")
print(f"Outage Statistics:")
print(f"  Mean: {np.nanmean(outage_data):.2f}")
print(f"  Std: {np.nanstd(outage_data):.2f}")
print(f"  Min: {np.nanmin(outage_data):.2f}")
print(f"  Max: {np.nanmax(outage_data):.2f}")

In [ ]:
# Load test datasets
print("Loading test datasets...")

test_24h_timestamps = ds_test_24h.timestamp.values
test_48h_timestamps = ds_test_48h.timestamp.values

print(f"✓ Test 24h: {len(test_24h_timestamps)} timestamps")
print(f"✓ Test 48h: {len(test_48h_timestamps)} timestamps")

print(f"\nTesting Period (24h): {test_24h_timestamps.min()} to {test_24h_timestamps.max()}")
print(f"Testing Period (48h): {test_48h_timestamps.min()} to {test_48h_timestamps.max()}")

In [ ]:
# sanity check on outage being positive and smaller than total tracked household
print(bool((ds_train.out <= ds_train.tracked).all()))
print(bool((ds_train.out>=0).all()))
print(bool((ds_train.tracked>=0).all()))

## 3. Exploratory Data Analysis

EDA is important for you to understand the data and the problem. Here I just show some basics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) Compute population per county
pop_by_loc = ds_train.tracked.mean(dim="timestamp")

# 2) Get top 10 locations by population
top5_locs = (
    pop_by_loc
    .sortby(pop_by_loc, ascending=False)
    .isel(location=slice(0, 5))
    .location
    .values
)

print("Top 5 locations by population:", [str(x) for x in top5_locs])

# 3) Plot their outages over time
fig, ax = plt.subplots(figsize=(15, 6))

for loc in top5_locs:
    outages = ds_train.out.sel(location=loc).values
    ax.plot(train_timestamps, outages, label=str(loc), alpha=0.8, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Power Outages", fontsize=12)
ax.set_title("Power Outages Over Time — Top 10 Counties by Population",
             fontsize=14, fontweight="bold")
ax.legend(ncol=2, fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Get the feature index for t2m
t2m_idx = weather_features.index('t2m')

fig, ax = plt.subplots(figsize=(15, 6))

# Plot t2m for the top 5 counties
for loc in top5_locs:
    # Extract t2m data for this location: (timestamp, feature) -> select t2m
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    ax.plot(train_timestamps, t2m_data, label=str(loc), alpha=0.8, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Temperature at 2m (K)", fontsize=12)
ax.set_title("Temperature (t2m) Over Time — Top 5 Counties by Population",
                fontsize=14, fontweight="bold")
ax.legend(ncol=2, fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print some statistics
print("\nt2m Statistics for Top 5 Counties:")
for loc in top5_locs:
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    print(f"  {loc}: Mean={np.mean(t2m_data):.2f}K, "
            f"Min={np.min(t2m_data):.2f}K, Max={np.max(t2m_data):.2f}K")

In [ ]:
print("Correlation Analysis: t2m (Temperature) vs. Power Outages\n")
print("="*70)

correlations = {}

for loc in top5_locs:
    # Get t2m data
    t2m_data = ds_train.weather.sel(location=loc, feature='t2m').values
    # Get outage data
    outage_data_loc = ds_train.out.sel(location=loc).values
    
    # Remove NaN values for correlation calculation
    valid_mask = ~(np.isnan(t2m_data) | np.isnan(outage_data_loc))
    t2m_clean = t2m_data[valid_mask]
    outage_clean = outage_data_loc[valid_mask]
    
    # Calculate Pearson correlation
    if len(t2m_clean) > 1:
        correlation = np.corrcoef(t2m_clean, outage_clean)[0, 1]
        correlations[str(loc)] = correlation
        
        direction = "Positive" if correlation > 0 else "Negative"
        
        print(f"County {loc}:")
        print(f"  Correlation: {correlation:+.4f} ({direction})")
        print(f"  Sample size: {len(t2m_clean)} valid data points")
        print()
    else:
        print(f"County {loc}: Insufficient data for correlation")
        correlations[str(loc)] = np.nan

# Visualize correlations
fig, ax = plt.subplots(figsize=(10, 6))

counties_list = list(correlations.keys())
corr_values = list(correlations.values())
colors = ['red' if c < 0 else 'blue' for c in corr_values]

bars = ax.bar(counties_list, corr_values, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('County', fontsize=12)
ax.set_ylabel('Correlation Coefficient', fontsize=12)
ax.set_title('Correlation between Temperature (t2m) and Power Outages\nTop 5 Counties by Population', 
                fontsize=14, fontweight='bold')
ax.set_ylim(-1, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, corr_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + (0.05 if height > 0 else -0.05),
            f'{val:.3f}', ha='center', va='bottom' if height > 0 else 'top', 
            fontsize=10, fontweight='bold')


## 4. Data Preparation

Split the training data into train and validation sets for model selection.

In [ ]:
# Create validation split (temporal split)
n_timestamps = len(train_timestamps)
split_idx = int(n_timestamps * (1 - VALIDATION_SPLIT))

# Split the dataset
ds_train_sub = ds_train.isel(timestamp=slice(0, split_idx))
ds_val = ds_train.isel(timestamp=slice(split_idx, None))

train_sub_timestamps = pd.to_datetime(ds_train_sub.timestamp.values)
val_timestamps = pd.to_datetime(ds_val.timestamp.values)

print(f"Training Subset: {len(train_sub_timestamps)} timestamps")
print(f"  Period: {train_sub_timestamps.min()} to {train_sub_timestamps.max()}")
print(f"\nValidation Set: {len(val_timestamps)} timestamps")
print(f"  Period: {val_timestamps.min()} to {val_timestamps.max()}")

# Store validation ground truth for later evaluation
val_truth = ds_val.out.transpose("timestamp", "location").values.astype(float)

In [ ]:
# Prepare ground truth for last 24 and 48 hours to align with test set
val_truth_24h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 24)).values.astype(float)
val_truth_48h = ds_val.out.transpose("timestamp", "location").isel(timestamp=slice(0, 48)).values.astype(float)

print(f"\Validation set shapes:")
print(f"  24h: {val_truth_24h.shape}")
print(f"  48h: {val_truth_48h.shape}")

In [ ]:
test_24h_truth = ds_test_24h.out.transpose("timestamp", "location").values.astype(float)
test_48h_truth = ds_test_48h.out.transpose("timestamp", "location").values.astype(float)


print(f"\nTest shapes:")
print(f"  24h: {test_24h_truth.shape}")
print(f"  48h: {test_48h_truth.shape}")

## 5. Model Structure and Functions

We'll define two types of models:
1. **SARIMAX** - Statistical time series model (per-county)
2. **Seq2Seq LSTM** - Deep learning model (shared across counties but different for two horizons)

### 5.1 SARIMAX

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def safe_fit_sarimax(y, order=(1, 0, 1)):
    """Safely fit SARIMAX model with error handling."""
    y = np.asarray(y, dtype=float).flatten()
    if len(y) < 8 or np.allclose(y, y[0]):
        return None
    try:
        model = SARIMAX(y, order=order, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        return res
    except Exception as e:
        print(f"  Warning: SARIMAX fit failed - {str(e)[:50]}")
        return None


### 5.2 Seq2Seq

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import time

# Set random seed for PyTorch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# Define Seq2Seq Model
class SimpleSeq2Seq(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, horizon=24):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_dim, horizon)
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        _, (h, _) = self.lstm(x)
        h_last = h[-1]  # (batch, hidden_dim)
        return self.head(h_last)  # (batch, horizon)

print(f"Device: {DEVICE}")

In [ ]:
# Utility functions for Seq2Seq
def z_normalize_fit(arr):
    """Compute mean and std for normalization."""
    mu = np.nanmean(arr, axis=0)
    sd = np.nanstd(arr, axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    return mu, sd

def z_normalize_apply(arr, mu, sd):
    """Apply normalization with precomputed mean and std."""
    return (arr - mu) / sd

def build_sliding_windows(X_loc, y_loc, seq_len, horizon):
    """
    Build sliding windows for one location.
    X_loc: (T, D) features
    y_loc: (T,) targets
    Returns: X_windows (N, seq_len, D), Y_windows (N, horizon)
    """
    N = len(y_loc) - seq_len - horizon + 1
    if N <= 0:
        return np.empty((0, seq_len, X_loc.shape[1]), dtype=float), np.empty((0, horizon), dtype=float)
    
    X_windows, Y_windows = [], []
    for i in range(N):
        X_windows.append(X_loc[i:i+seq_len])
        Y_windows.append(y_loc[i+seq_len:i+seq_len+horizon])
    
    return np.array(X_windows, dtype=float), np.array(Y_windows, dtype=float)

print("Helper functions defined.")

In [ ]:
# Prepare training data for Seq2Seq (using training subset)
def prepare_seq2seq_data(ds, seq_len, horizon):
    """
    Prepare data for Seq2Seq training.
    Features: [outage_scaled, weather_features_scaled]
    """
    y = ds.out.transpose("timestamp", "location").values.astype(float)  # (T, L)
    w = ds.weather.transpose("timestamp", "location", "feature").values.astype(float)  # (T, L, F)
    T, L, F = w.shape
    
    # Compute global scalers
    y_mu, y_sd = z_normalize_fit(y.reshape(-1, 1))
    w_mu, w_sd = z_normalize_fit(w.reshape(-1, F))
    
    # Apply scaling
    y_scaled = z_normalize_apply(y.reshape(-1, 1), y_mu, y_sd).reshape(T, L)
    w_scaled = z_normalize_apply(w.reshape(-1, F), w_mu, w_sd).reshape(T, L, F)
    
    # Build windows for all locations
    input_dim = 1 + F  # outage + weather features
    X_list, Y_list = [], []
    
    for li in range(L):
        y_loc = y_scaled[:, li]  # (T,)
        w_loc = w_scaled[:, li, :]  # (T, F)
        X_loc = np.concatenate([y_loc.reshape(-1, 1), w_loc], axis=1)  # (T, 1+F)
        
        X_win, Y_win = build_sliding_windows(X_loc, y_loc, seq_len, horizon)
        if len(X_win) > 0:
            X_list.append(X_win)
            Y_list.append(Y_win)
    
    X = np.concatenate(X_list, axis=0) if X_list else np.empty((0, seq_len, input_dim))
    Y = np.concatenate(Y_list, axis=0) if Y_list else np.empty((0, horizon))
    
    scalers = {"y_mu": y_mu, "y_sd": y_sd, "w_mu": w_mu, "w_sd": w_sd}
    return X, Y, input_dim, scalers

# Use validation set length as horizon for training
val_horizon = len(val_timestamps)
print(f"Preparing Seq2Seq training data with horizon={val_horizon}...")

X_train, Y_train, input_dim, seq2seq_scalers = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, val_horizon)

print(f"Training windows created: {X_train.shape[0]} samples")
print(f"Input shape: {X_train.shape} (samples, seq_len, features)")
print(f"Output shape: {Y_train.shape} (samples, horizon)")
print(f"Input dimension: {input_dim}")

In [ ]:
# Train Seq2Seq model with optional validation RMSE monitoring
def train_seq2seq(X, Y, input_dim, horizon, epochs=5, batch_size=64, lr=1e-3,
                  X_val=None, Y_val=None, scalers=None):
    """
    Train Seq2Seq model.
    
    If X_val / Y_val are provided, computes and logs validation RMSE each epoch.
    If scalers dict (with y_mu, y_sd) is given, RMSE is reported in original scale
    (matching the competition metric).
    """
    if len(X) == 0:
        print("No training data available!")
        return None
    
    # Create dataset and dataloader
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(Y, dtype=torch.float32)
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Prepare validation tensors (if provided)
    has_val = X_val is not None and Y_val is not None and len(X_val) > 0
    if has_val:
        X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        Y_val_t = torch.tensor(Y_val, dtype=torch.float32).to(DEVICE)
        # For inverse-scaling to original units
        if scalers is not None:
            y_mu = float(scalers["y_mu"])
            y_sd = float(scalers["y_sd"])
        else:
            y_mu, y_sd = 0.0, 1.0  # no inverse scaling
    
    # Initialize model
    model = SimpleSeq2Seq(
        input_dim=input_dim,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        horizon=horizon
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    best_val_rmse = float("inf")
    
    print(f"\nTraining Seq2Seq model for {epochs} epochs...")
    if has_val:
        print(f"  Validation samples: {len(X_val)} (RMSE in original scale)")
    
    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        epoch_loss = 0.0
        start_time = time.time()
        
        for xb, yb in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            epoch_loss += loss.item() * xb.size(0)
        
        epoch_loss /= len(dataset)
        train_rmse_scaled = float(np.sqrt(epoch_loss))
        train_rmse = train_rmse_scaled * y_sd if has_val else train_rmse_scaled
        elapsed = time.time() - start_time
        
        # ---- Validation RMSE (competition metric proxy) ----
        val_rmse = None
        if has_val:
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t)  # (N, horizon) in scaled space
                val_mse = criterion(val_pred, Y_val_t).item()
                # RMSE in original scale: sqrt(MSE) * y_sd
                val_rmse = float(np.sqrt(val_mse)) * y_sd
                if val_rmse < best_val_rmse:
                    best_val_rmse = val_rmse
        
        # ---- Logging ----
        log_msg = f"Epoch {epoch+1}/{epochs} - Train RMSE: {train_rmse:.4f}"
        if val_rmse is not None:
            log_msg += f" - Val RMSE: {val_rmse:.4f} (best: {best_val_rmse:.4f})"
        log_msg += f" - Time: {elapsed:.1f}s"
        print(log_msg)
        
        if WANDB_ENABLED:
            log_dict = {
                f"seq2seq_{horizon}h/train_loss": epoch_loss,
                f"seq2seq_{horizon}h/train_rmse": train_rmse,
                f"seq2seq_{horizon}h/epoch": epoch + 1,
                f"seq2seq_{horizon}h/epoch_time": elapsed,
            }
            if val_rmse is not None:
                log_dict[f"seq2seq_{horizon}h/val_rmse"] = val_rmse
                log_dict[f"seq2seq_{horizon}h/best_val_rmse"] = best_val_rmse
            wandb.log(log_dict)
    
    if has_val:
        print(f"  ✓ Best validation RMSE: {best_val_rmse:.4f}")
    
    return model

### 5.3 Prediction Functions

Define reusable prediction functions that will be used for both validation and test sets.

In [ ]:
# Consolidated prediction functions for reuse

def generate_sarimax_predictions(models_dict, locations, timestamps):
    """
    Generate SARIMAX predictions for given timestamps.
    Returns: DataFrame in long format (timestamp, location, pred)
    """
    rows = []
    n_steps = len(timestamps)
    
    for loc in locations:
        loc_str = str(loc)
        if loc_str in models_dict and models_dict[loc_str] is not None:
            try:
                pred = np.asarray(models_dict[loc_str].forecast(steps=n_steps), dtype=float)
                pred = np.clip(pred, 0, None)  # Non-negative constraint
            except:
                pred = np.zeros(n_steps)
        else:
            pred = np.zeros(n_steps)
        
        rows.append(pd.DataFrame({
            "timestamp": timestamps,
            "location": loc_str,
            "pred": pred
        }))
    
    return pd.concat(rows, ignore_index=True)

@torch.no_grad()
def generate_seq2seq_predictions(model, ds_train_data, scalers, horizon, timestamps, locations):
    """
    Generate Seq2Seq predictions for given horizon.
    Returns: DataFrame in long format (timestamp, location, pred)
    """
    if model is None:
        # Return zeros if model not available
        locs_repeated = np.repeat([str(loc) for loc in locations], horizon)
        ts_repeated = np.tile(timestamps, len(locations))
        return pd.DataFrame({
            "timestamp": ts_repeated,
            "location": locs_repeated,
            "pred": 0.0
        })
    
    model.eval()
    
    y = ds_train_data.out.transpose("timestamp", "location").values.astype(float)
    w = ds_train_data.weather.transpose("timestamp", "location", "feature").values.astype(float)
    T, L, F = w.shape
    
    # Apply scaling
    y_scaled = z_normalize_apply(y.reshape(-1, 1), scalers["y_mu"], scalers["y_sd"]).reshape(T, L)
    w_scaled = z_normalize_apply(w.reshape(-1, F), scalers["w_mu"], scalers["w_sd"]).reshape(T, L, F)
    
    predictions = []
    
    for li in range(L):
        if T < SEQ_LEN:
            pred_scaled = np.zeros(horizon)
        else:
            # Get last SEQ_LEN timesteps
            y_loc = y_scaled[:, li]
            w_loc = w_scaled[:, li, :]
            X_loc = np.concatenate([y_loc.reshape(-1, 1), w_loc], axis=1)
            X_in = X_loc[-SEQ_LEN:].reshape(1, SEQ_LEN, -1)
            
            # Predict
            X_tensor = torch.tensor(X_in, dtype=torch.float32).to(DEVICE)
            pred_scaled = model(X_tensor).cpu().numpy()[0]
        
        # Inverse scaling
        pred = (pred_scaled * scalers["y_sd"].flatten()[0]) + scalers["y_mu"].flatten()[0]
        pred = np.clip(pred, 0, None)  # Non-negative constraint
        predictions.append(pred)
    
    # Convert to DataFrame in long format
    preds_array = np.array(predictions).T  # (horizon, L)
    rows = []
    for i, loc in enumerate(locations):
        rows.append(pd.DataFrame({
            "timestamp": timestamps,
            "location": str(loc),
            "pred": preds_array[:, i]
        }))
    
    return pd.concat(rows, ignore_index=True)

### 5.4 Evaluation Metric Fucntions

In [ ]:
# Define evaluation metrics
def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# def mae(y_true, y_pred):
#     """Calculate Mean Absolute Error."""
#     y_true = np.asarray(y_true, dtype=float)
#     y_pred = np.asarray(y_pred, dtype=float)
#     return float(np.mean(np.abs(y_true - y_pred)))

## 6. Validation Set (24h and 48h)

Evaluate models on 24h and 48h prediction horizons separately, matching the test set structure.

In [ ]:
# Define horizons
horizon_24h_val = 24
horizon_48h_val = 48

# Extract validation timestamps for each horizon
val_timestamps_24h = val_timestamps[:24]
val_timestamps_48h = val_timestamps[:48]

### 6.1 Train Models

In [ ]:
# Train SARIMAX models per county on training subset
print("Training SARIMAX models (one per county)...\n")
sarimax_models = {}

# Notice we fixed SARIMAX_ORDER for all counties, however, one can easily tune it over validation set for each county.
for loc in locations:
    print(f"Fitting SARIMAX for {loc}...", end=" ")
    y_train = ds_train_sub.out.sel(location=loc).values.astype(float).flatten()
    model = safe_fit_sarimax(y_train, order=SARIMAX_ORDER)
    sarimax_models[loc] = model
    if model is not None:
        print(f"✓ Success (AIC: {model.aic:.2f})")
    else:
        print("✗ Failed (will use zero baseline)")

print(f"\nSARIMAX training complete!")

In [ ]:
# Train Seq2Seq models per county on training subset
# Also prepare validation windows for in-training RMSE monitoring
print("Training SARIMAX models (one per time horizon for all counties)...\n")
print(f"\nValidation horizons:")
print(f"  24h: {len(val_timestamps_24h)} timestamps")
print(f"  48h: {len(val_timestamps_48h)} timestamps")

# Prepare validation windows from the held-out validation portion
# so we can monitor RMSE *during* training (not just after)
print("\nPreparing validation windows for in-training RMSE monitoring...")
X_val_24h, Y_val_24h, _, _ = prepare_seq2seq_data(ds_val, SEQ_LEN, horizon_24h_val)
X_val_48h, Y_val_48h, _, _ = prepare_seq2seq_data(ds_val, SEQ_LEN, horizon_48h_val)
print(f"  24h val windows: {X_val_24h.shape[0]}")
print(f"  48h val windows: {X_val_48h.shape[0]}")

# Train for 24h horizon
print(f"\n{'='*70}")
print("Training Seq2Seq for 24h horizon...")
print(f"{'='*70}")
X_train_24h, Y_train_24h, input_dim, seq2seq_scalers_24h = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, horizon_24h_val)
print(f"Training samples: {X_train_24h.shape[0]}")

seq2seq_model_24h_val = train_seq2seq(
    X_train_24h, Y_train_24h, input_dim, horizon_24h_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
    X_val=X_val_24h, Y_val=Y_val_24h, scalers=seq2seq_scalers_24h
)

# Train for 48h horizon
print(f"\n{'='*70}")
print("Training Seq2Seq for 48h horizon...")
print(f"{'='*70}")
X_train_48h, Y_train_48h, _, seq2seq_scalers_48h = prepare_seq2seq_data(ds_train_sub, SEQ_LEN, horizon_48h_val)
print(f"Training samples: {X_train_48h.shape[0]}")

seq2seq_model_48h_val = train_seq2seq(
    X_train_48h, Y_train_48h, input_dim, horizon_48h_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
    X_val=X_val_48h, Y_val=Y_val_48h, scalers=seq2seq_scalers_48h
)

print("\n✓ Seq2Seq models trained for both horizons!")

### 6.2 Generate Predictions

In [ ]:
# Generate predictions using consolidated functions
print("Generating predictions for validation set...\n")

# SARIMAX predictions (same models for both horizons, just different forecast lengths)
print("1) SARIMAX predictions...")
sarimax_val_pred_24h_df = generate_sarimax_predictions(sarimax_models, locations, val_timestamps_24h)
sarimax_val_pred_48h_df = generate_sarimax_predictions(sarimax_models, locations, val_timestamps_48h)
print(f"   24h: {len(sarimax_val_pred_24h_df)} predictions")
print(f"   48h: {len(sarimax_val_pred_48h_df)} predictions")

# Seq2Seq predictions
print("\n2) Seq2Seq predictions...")
seq2seq_val_pred_24h_df = generate_seq2seq_predictions(
    seq2seq_model_24h_val, ds_train_sub, seq2seq_scalers_24h, 
    horizon_24h_val, val_timestamps_24h, locations
)
seq2seq_val_pred_48h_df = generate_seq2seq_predictions(
    seq2seq_model_48h_val, ds_train_sub, seq2seq_scalers_48h,
    horizon_48h_val, val_timestamps_48h, locations
)
print(f"   24h: {len(seq2seq_val_pred_24h_df)} predictions")
print(f"   48h: {len(seq2seq_val_pred_48h_df)} predictions")

print("\n✓ All predictions generated!")

### 6.3 Evaluate Performance on Validation Set

In [ ]:

# Helper function to evaluate per-county RMSE
def evaluate_per_county(truth, pred_df, locations):
    """Evaluate RMSE per county and return list of RMSEs"""
    rmses = []
    for i, loc in enumerate(locations):
        loc_str = str(loc)
        loc_pred = pred_df[pred_df['location'].astype(str) == loc_str]['pred'].values
        if len(loc_pred) == truth.shape[0]:
            county_rmse = rmse(truth[:, i], loc_pred)
            rmses.append(county_rmse)
        else:
            rmses.append(np.nan)
    return rmses

# Evaluate 24h horizon
print("="*70)
print("VALIDATION EVALUATION - 24H HORIZON")
print("="*70)

sarimax_24h_rmses = evaluate_per_county(val_truth_24h, sarimax_val_pred_24h_df, locations)
seq2seq_24h_rmses = evaluate_per_county(val_truth_24h, seq2seq_val_pred_24h_df, locations)
zero_24h_rmses = [rmse(val_truth_24h[:, i], np.zeros(24)) for i in range(len(locations))]

sarimax_24h_avg = np.nanmean(sarimax_24h_rmses)
seq2seq_24h_avg = np.nanmean(seq2seq_24h_rmses)
zero_24h_avg = np.nanmean(zero_24h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_24h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_24h_avg:.4f}")
print(f"  Zero Baseline: {zero_24h_avg:.4f}")

sarimax_24h_imp = ((zero_24h_avg - sarimax_24h_avg) / zero_24h_avg * 100)
seq2seq_24h_imp = ((zero_24h_avg - seq2seq_24h_avg) / zero_24h_avg * 100)

print(f"\nBest Model for 24h: ", end="")
if sarimax_24h_avg < seq2seq_24h_avg and sarimax_24h_avg < zero_24h_avg:
    print(f"✓ SARIMAX ({sarimax_24h_avg:.4f})")
elif seq2seq_24h_avg < sarimax_24h_avg and seq2seq_24h_avg < zero_24h_avg:
    print(f"✓ Seq2Seq ({seq2seq_24h_avg:.4f})")
else:
    print(f"⚠ Zero Baseline is better ({zero_24h_avg:.4f})")

# Evaluate 48h horizon
print("\n" + "="*70)
print("VALIDATION EVALUATION - 48H HORIZON")
print("="*70)

sarimax_48h_rmses = evaluate_per_county(val_truth_48h, sarimax_val_pred_48h_df, locations)
seq2seq_48h_rmses = evaluate_per_county(val_truth_48h, seq2seq_val_pred_48h_df, locations)
zero_48h_rmses = [rmse(val_truth_48h[:, i], np.zeros(48)) for i in range(len(locations))]

sarimax_48h_avg = np.nanmean(sarimax_48h_rmses)
seq2seq_48h_avg = np.nanmean(seq2seq_48h_rmses)
zero_48h_avg = np.nanmean(zero_48h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_48h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_48h_avg:.4f}")
print(f"  Zero Baseline: {zero_48h_avg:.4f}")

sarimax_48h_imp = ((zero_48h_avg - sarimax_48h_avg) / zero_48h_avg * 100)
seq2seq_48h_imp = ((zero_48h_avg - seq2seq_48h_avg) / zero_48h_avg * 100)

print(f"\nBest Model for 48h: ", end="")
if sarimax_48h_avg < seq2seq_48h_avg and sarimax_48h_avg < zero_48h_avg:
    print(f"✓ SARIMAX ({sarimax_48h_avg:.4f})")
elif seq2seq_48h_avg < sarimax_48h_avg and seq2seq_48h_avg < zero_48h_avg:
    print(f"✓ Seq2Seq ({seq2seq_48h_avg:.4f})")
else:
    print(f"⚠ Zero Baseline is better ({zero_48h_avg:.4f})")

# Log validation metrics to wandb
if WANDB_ENABLED:
    wandb.log({
        "val/sarimax_rmse_24h": sarimax_24h_avg,
        "val/seq2seq_rmse_24h": seq2seq_24h_avg,
        "val/zero_baseline_rmse_24h": zero_24h_avg,
        "val/sarimax_rmse_48h": sarimax_48h_avg,
        "val/seq2seq_rmse_48h": seq2seq_48h_avg,
        "val/zero_baseline_rmse_48h": zero_48h_avg,
    })

### 6.4 Visualize Validation Predictions (Top 5 Counties)

In [ ]:
# Visualize predictions for top 5 counties (24h horizon)
print("Visualizing 24h validation predictions for top 5 counties...\n")

fig, axes = plt.subplots(len(top5_locs), 1, figsize=(15, 4*len(top5_locs)))
if len(top5_locs) == 1:
    axes = [axes]

for plot_idx, loc in enumerate(top5_locs):
    ax = axes[plot_idx]
    loc_str = str(loc)
    loc_idx = locations.index(loc_str)
    
    # Get predictions for this location
    sarimax_pred = sarimax_val_pred_24h_df[sarimax_val_pred_24h_df['location'] == loc_str]['pred'].values
    seq2seq_pred = seq2seq_val_pred_24h_df[seq2seq_val_pred_24h_df['location'] == loc_str]['pred'].values
    
    ax.plot(val_timestamps_24h, val_truth_24h[:, loc_idx], label='Actual', 
            color='black', linewidth=2.5, marker='o', markersize=5)
    ax.plot(val_timestamps_24h, sarimax_pred, label='SARIMAX', 
            alpha=0.7, linewidth=2, marker='s', markersize=4)
    ax.plot(val_timestamps_24h, seq2seq_pred, label='Seq2Seq', 
            alpha=0.7, linewidth=2, marker='^', markersize=4)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Zero Baseline')
    
    ax.set_title(f'24h Validation: County {loc}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Outages')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Showing 24h validation predictions for: {[str(loc) for loc in top5_locs]}")

## 7. Test Set Prediction and Submission Generation (24h and 48h)

Train final models on full training data and evaluate on test sets (24h and 48h).

### 7.1 Train Final Models on Full Training Data

In [ ]:
# Train final SARIMAX models on full training data
print("="*70)
print("TRAINING FINAL SARIMAX MODELS (on full training data)")
print("="*70)

sarimax_final_models = {}
for loc in locations:
    loc_str = str(loc)
    y_full = ds_train.out.sel(location=loc).values.astype(float)
    model = safe_fit_sarimax(y_full, order=SARIMAX_ORDER)
    sarimax_final_models[loc_str] = model
    status = "✓" if model is not None else "✗"
    print(f"  {status} County {loc_str}")

print("\n✓ SARIMAX models trained!")


In [ ]:
# Train final Seq2Seq models for 24h and 48h
print("\n" + "="*70)
print("TRAINING FINAL SEQ2SEQ MODELS (on full training data)")
print("="*70)

# 24h model
print("\n1) Training Seq2Seq for 24h horizon...")
X_final_24h, Y_final_24h, input_dim_final, seq2seq_final_scalers_24h = prepare_seq2seq_data(ds_train, SEQ_LEN, 24)
print(f"   Training samples: {X_final_24h.shape[0]}")

seq2seq_final_model_24h = train_seq2seq(
    X_final_24h, Y_final_24h, input_dim_final, 24,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE
)

# 48h model
print("\n2) Training Seq2Seq for 48h horizon...")
X_final_48h, Y_final_48h, _, seq2seq_final_scalers_48h = prepare_seq2seq_data(ds_train, SEQ_LEN, 48)
print(f"   Training samples: {X_final_48h.shape[0]}")

seq2seq_final_model_48h = train_seq2seq(
    X_final_48h, Y_final_48h, input_dim_final, 48,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE
)

print("\n✓ All final models trained!")

### 7.2 Generate Predictions

In [ ]:
# Generate test predictions using the SAME consolidated functions as validation
print("Generating test set predictions...\n")

# 24h predictions
print("1) Test 24h predictions:")
sarimax_test_24h_df = generate_sarimax_predictions(sarimax_final_models, locations, test_24h_timestamps)
seq2seq_test_24h_df = generate_seq2seq_predictions(
    seq2seq_final_model_24h, ds_train, seq2seq_final_scalers_24h,
    24, test_24h_timestamps, locations
)
print(f"   SARIMAX: {len(sarimax_test_24h_df)} predictions")
print(f"   Seq2Seq: {len(seq2seq_test_24h_df)} predictions")

# 48h predictions
print("\n2) Test 48h predictions:")
sarimax_test_48h_df = generate_sarimax_predictions(sarimax_final_models, locations, test_48h_timestamps)
seq2seq_test_48h_df = generate_seq2seq_predictions(
    seq2seq_final_model_48h, ds_train, seq2seq_final_scalers_48h,
    48, test_48h_timestamps, locations
)
print(f"   SARIMAX: {len(sarimax_test_48h_df)} predictions")
print(f"   Seq2Seq: {len(seq2seq_test_48h_df)} predictions")

print("\n✓ All test predictions generated!")

### 7.3 Evaluate Perofrmance on Test Set (demo only)

The following code is just to verify if your submission files by your model can be evaluated properly or if you followed the format of prediciont file. It is using demo test files, ```test_24h_demo.nc``` and ```test_48h_demo.nc```. Again these files are just randome noise but wtih the right shape and format. So the test results say nothing about the final evaluation.

In [ ]:
# Evaluate 24h test set
print("="*70)
print("TEST EVALUATION - 24H HORIZON")
print("="*70)

sarimax_test_24h_rmses = evaluate_per_county(test_24h_truth, sarimax_test_24h_df, locations)
seq2seq_test_24h_rmses = evaluate_per_county(test_24h_truth, seq2seq_test_24h_df, locations)
zero_test_24h_rmses = [rmse(test_24h_truth[:, i], np.zeros(24)) for i in range(len(locations))]

sarimax_test_24h_avg = np.nanmean(sarimax_test_24h_rmses)
seq2seq_test_24h_avg = np.nanmean(seq2seq_test_24h_rmses)
zero_test_24h_avg = np.nanmean(zero_test_24h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_test_24h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_test_24h_avg:.4f}")
print(f"  Zero Baseline: {zero_test_24h_avg:.4f}")

sarimax_test_24h_imp = ((zero_test_24h_avg - sarimax_test_24h_avg) / zero_test_24h_avg * 100)
seq2seq_test_24h_imp = ((zero_test_24h_avg - seq2seq_test_24h_avg) / zero_test_24h_avg * 100)

# Evaluate 48h test set
print("\n" + "="*70)
print("TEST EVALUATION - 48H HORIZON")
print("="*70)

sarimax_test_48h_rmses = evaluate_per_county(test_48h_truth, sarimax_test_48h_df, locations)
seq2seq_test_48h_rmses = evaluate_per_county(test_48h_truth, seq2seq_test_48h_df, locations)
zero_test_48h_rmses = [rmse(test_48h_truth[:, i], np.zeros(48)) for i in range(len(locations))]

sarimax_test_48h_avg = np.nanmean(sarimax_test_48h_rmses)
seq2seq_test_48h_avg = np.nanmean(seq2seq_test_48h_rmses)
zero_test_48h_avg = np.nanmean(zero_test_48h_rmses)

print(f"\nAverage RMSE across counties:")
print(f"  SARIMAX: {sarimax_test_48h_avg:.4f}")
print(f"  Seq2Seq: {seq2seq_test_48h_avg:.4f}")
print(f"  Zero Baseline: {zero_test_48h_avg:.4f}")

sarimax_test_48h_imp = ((zero_test_48h_avg - sarimax_test_48h_avg) / zero_test_48h_avg * 100)
seq2seq_test_48h_imp = ((zero_test_48h_avg - seq2seq_test_48h_avg) / zero_test_48h_avg * 100)

# Log test metrics to wandb
if WANDB_ENABLED:
    wandb.log({
        "test/sarimax_rmse_24h": sarimax_test_24h_avg,
        "test/seq2seq_rmse_24h": seq2seq_test_24h_avg,
        "test/zero_baseline_rmse_24h": zero_test_24h_avg,
        "test/sarimax_rmse_48h": sarimax_test_48h_avg,
        "test/seq2seq_rmse_48h": seq2seq_test_48h_avg,
        "test/zero_baseline_rmse_48h": zero_test_48h_avg,
    })
    wandb.finish()
    print("\n✓ W&B run finished. Check your dashboard for results.")

### 7.4 Save Predictions to CSV

In [ ]:
# Save predictions to CSV files for submission
print("Saving predictions to CSV files...\n")

# Save SARIMAX predictions
sarimax_test_24h_df.to_csv(os.path.join(RESULTS_DIR, "sarimax_pred_24h.csv"), index=False)
sarimax_test_48h_df.to_csv(os.path.join(RESULTS_DIR, "sarimax_pred_48h.csv"), index=False)
print("✓ SARIMAX predictions saved:")
print(f"  - {os.path.join(RESULTS_DIR, 'sarimax_pred_24h.csv')}")
print(f"  - {os.path.join(RESULTS_DIR, 'sarimax_pred_48h.csv')}")

# Save Seq2Seq predictions
seq2seq_test_24h_df.to_csv(os.path.join(RESULTS_DIR, "seq2seq_pred_24h.csv"), index=False)
seq2seq_test_48h_df.to_csv(os.path.join(RESULTS_DIR, "seq2seq_pred_48h.csv"), index=False)
print("\n✓ Seq2Seq predictions saved:")
print(f"  - {os.path.join(RESULTS_DIR, 'seq2seq_pred_24h.csv')}")
print(f"  - {os.path.join(RESULTS_DIR, 'seq2seq_pred_48h.csv')}")

print("\n" + "="*70)
print("All predictions saved successfully!")
print("="*70)


### 7.5 Evaluate Performance on Test Set

We will be evluating your submitted files following similar procedure in this part but with actual test files for your predictive performance.

You should run these to check your saved files works with these code.

In [ ]:
your_24hr_prediction_filepath = "results/sarimax_pred_24h.csv"
your_48hr_prediction_filepath = "results/sarimax_pred_48h.csv"
# your_24hr_prediction_filepath = "results/seq2seq_pred_24h.csv"
# your_48hr_prediction_filepath = "results/seq2seq_pred_48h.csv"

# load your prediction file
df_24 = pd.read_csv(your_24hr_prediction_filepath)
df_48 = pd.read_csv(your_48hr_prediction_filepath)

In [ ]:
# You prediction file should have three columns: timestamp, location, pred
# and number of rows should be 24*83 = 1992 or 48*83 = 3984 for 24h and 48h prediction respectively

# check the shape
assert df_24.shape == (1992,3)
assert df_48.shape == (3984,3)


# check the column names
assert df_24.columns.tolist() == ['timestamp', 'location', 'pred']
assert df_48.columns.tolist() == ['timestamp', 'location', 'pred']

In [ ]:
# Just making sure, evlauate on test set again.
test_rmses_24 = evaluate_per_county(test_24h_truth, df_24, locations)
test_rmses_48 = evaluate_per_county(test_48h_truth, df_48, locations)
test_rmses_avg_24 = np.nanmean(test_rmses_24)
test_rmses_avg_48 = np.nanmean(test_rmses_48)

print(f"test_24h_rmses: {test_rmses_avg_24}")
print(f"test_48h_rmses: {test_rmses_avg_48}")